In [13]:
from datetime import datetime

import pandas as pd
import requests

calendar_feed_url = 'https://calendar.google.com/calendar/ical/calendar%40iitmandi.ac.in/public/basic.ics'
response = requests.get(calendar_feed_url, timeout=30)
response.raise_for_status()

def parse_ics_datetime(value: str) -> str:
    if 'T' in value:
        if value.endswith('Z'):
            return datetime.strptime(value, '%Y%m%dT%H%M%SZ').strftime('%Y-%m-%d %H:%M')
        return datetime.strptime(value, '%Y%m%dT%H%M%S').strftime('%Y-%m-%d %H:%M')
    return datetime.strptime(value, '%Y%m%d').strftime('%Y-%m-%d')

events = []
current_event = None
in_event = False

for raw_line in response.text.splitlines():
    line = raw_line.strip()

    if line == 'BEGIN:VEVENT':
        in_event = True
        current_event = {}
        continue

    if line == 'END:VEVENT':
        if current_event and current_event.get('date') and current_event.get('event_title'):
            events.append(current_event)
        in_event = False
        current_event = None
        continue

    if not in_event or ':' not in line:
        continue

    if line.startswith('DTSTART'):
        value = line.split(':', 1)[1]
        current_event['date'] = parse_ics_datetime(value)
        current_event['event_time'] = 'All day' if 'VALUE=DATE' in line else current_event['date'].split(' ', 1)[1]
    elif line.startswith('SUMMARY:'):
        current_event['event_title'] = line[len('SUMMARY:'):].strip()

df = pd.DataFrame(events, columns=['date', 'event_time', 'event_title'])
df.to_csv('IIT_MANDI_CALENDAR.csv', index=False)
df.head()

,date,event_time,event_title
0,2024-03-12,All day,Mid Sem E & F
1,2026-12-02,All day,End Sem E
2,2025-07-23,All day,Summer End Sem / Supplementary Exams
3,2026-11-17,All day,Final TCF Submission (upto 28 Nov)
4,2024-09-25,All day,Mid sem C&D
